# Partie 4 — Entraînement & optimisation des modèles

**Projet :** prédiction du churn client (Telco Customer Churn) — groupe de 5
**Responsable :** Membre 4
**Livrable attendu :** modèles entraînés + tableau comparatif des expériences

## Objectif de cette partie

Les parties 1 à 3 ont fourni un dataset compris, exploré et préparé. Ici, on **entraîne plusieurs modèles de classification**, on les compare de façon **honnête** (sans fuite de données) et on **optimise** les meilleurs. L'évaluation finale détaillée (matrices de confusion, interprétation) est le travail de la partie 5.

## Ce qu'on retient des parties précédentes

| Constat | Conséquence pour la modélisation |
|---|---|
| 26,5 % de churn seulement (classes déséquilibrées) | l'accuracy est trompeuse : un modèle « toujours No » obtient 73,5 %. On suit surtout **Recall, F1 et ROC-AUC** de la classe churn |
| Contrat, ancienneté, fibre, chèque électronique = facteurs de risque | on pourra vérifier que les modèles retrouvent ces variables |
| `tenure`, `TotalCharges`, `MonthlyCharges` très corrélées | attention aux modèles linéaires (régularisation) |
| Découpage train/test déjà fait (80/20, stratifié, `random_state=42`) | on **réutilise exactement le même découpage** ; le test n'est touché qu'à la toute fin |

## Plan du notebook

1. **Mise en place** : configuration, chargement des données, reconstruction du pipeline de prétraitement
2. Stratégie de validation et modèle de référence
3. Entraînement de plusieurs modèles (Logistic Regression, Decision Tree, Random Forest…)
4. Gestion du déséquilibre des classes
5. Optimisation des hyperparamètres
6. Tableau comparatif des expériences et sauvegarde des modèles

## 1. Mise en place

### 1.1 Imports et configuration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import imblearn
import joblib

# Graine unique pour toute la partie 4 : mêmes résultats à chaque exécution
SEED = 42

# Dossier lu par la partie 4 (produit par la partie 3) et dossiers de sortie
OUT_DIR = Path("outputs")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")

pd.set_option("display.max_columns", 60)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__}")

Python 3.13.7 | pandas 3.1.1 | numpy 2.5.4
scikit-learn 1.9.1 | imbalanced-learn 0.14.2


**Pourquoi afficher les versions ?** Un modèle sauvegardé avec `joblib` ne se recharge correctement qu'avec la même version de scikit-learn. Les versions sont fixées dans `requirements.txt` ; cette cellule permet de vérifier qu'on travaille avec le même environnement que le reste du groupe.

### 1.2 Chargement des données

On part de `telco_clean_features.csv` : les données **nettoyées avec les nouvelles variables, mais avant encodage et mise à l'échelle** (produit par la partie 3).

**Pourquoi ne pas utiliser directement `train_processed.csv` ?** Dans ces fichiers, l'imputation et la standardisation ont déjà été apprises sur *tout* le train. Si on fait ensuite une validation croisée dessus, les statistiques (moyenne, écart-type) de chaque pli de validation ont déjà « fuité » dans l'entraînement : c'est une fuite légère, mais réelle. En repartant des données avant encodage, on pourra remettre le prétraitement **à l'intérieur** de chaque pli de validation croisée.

In [2]:
df = pd.read_csv(OUT_DIR / "telco_clean_features.csv")
print("Dimensions :", df.shape)
print("Valeurs manquantes par colonne (uniquement celles > 0) :")
print(df.isna().sum()[lambda s: s > 0])
df.head(3)

Dimensions : (7043, 45)
Valeurs manquantes par colonne (uniquement celles > 0) :
Series([], dtype: int64)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,is_new_customer,tenure_years,has_internet,has_phone,n_security_services,n_streaming_services,n_services_total,has_protection_bundle,internet_no_protection,avg_monthly_spend,charge_delta,charge_ratio,charge_per_service,log_total_charges,is_month_to_month,contract_length_m,auto_payment,is_electronic_check,has_family,is_alone,fiber_month_to_month,new_and_mtm,senior_alone
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-6m,1,0.083333,1,0,1,0,2,0,0,29.850000,0.000000,1.000000,14.9250,3.429137,1,1,0,1,1,0,0,1,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,25-48m,0,2.833333,1,1,2,0,4,0,0,55.573529,1.376471,1.024768,14.2375,7.544597,0,12,0,0,0,1,0,0,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-6m,1,0.166667,1,1,2,0,4,0,0,54.075000,-0.225000,0.995839,13.4625,4.692723,1,1,0,0,0,1,0,1,0


### 1.3 Découpage train / test (identique à la partie 3)

Mêmes paramètres que la partie 3 : 80 % / 20 %, stratifié sur `Churn`, `random_state=42`. Le **jeu de test est mis de côté** : il ne servira qu'à la fin, pour évaluer le modèle retenu. Toutes les décisions (choix du modèle, réglage des hyperparamètres) se feront uniquement avec le train.

In [3]:
from sklearn.model_selection import train_test_split

TARGET = "Churn"
X = df.drop(columns=[TARGET, "customerID"])   # customerID est un identifiant, pas une variable explicative
y = df[TARGET]
ids = df["customerID"]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.20, stratify=y, random_state=SEED)

print(f"Train : {X_train.shape} | churn = {y_train.mean():.2%}")
print(f"Test  : {X_test.shape}  | churn = {y_test.mean():.2%}")

Train : (5634, 43) | churn = 26.54%
Test  : (1409, 43)  | churn = 26.54%


### 1.4 Vérification : est-ce le même découpage que la partie 3 ?

Si le groupe n'utilise pas les mêmes clients dans le train et le test, les résultats des parties 4 et 5 ne seront pas comparables. On compare donc les identifiants avec les fichiers produits par la partie 3.

In [4]:
ref_train = pd.read_csv(OUT_DIR / "train_processed.csv", usecols=["customerID", "Churn"])
ref_test = pd.read_csv(OUT_DIR / "test_processed.csv", usecols=["customerID", "Churn"])

assert set(id_train) == set(ref_train["customerID"]), "Le train n'est pas celui de la partie 3"
assert set(id_test) == set(ref_test["customerID"]), "Le test n'est pas celui de la partie 3"
assert set(id_train).isdisjoint(set(id_test)), "Des clients sont à la fois dans le train et le test"
print("OK : train et test identiques à ceux de la partie 3 "
      f"({len(id_train)} + {len(id_test)} clients, aucun client en commun).")

OK : train et test identiques à ceux de la partie 3 (5634 + 1409 clients, aucun client en commun).


**Résultat :** on travaille bien sur les mêmes clients que le reste du groupe, avec la même proportion de churn (26,54 %) dans le train et dans le test.